# Фракталы как вычислительный эксперимент

Продолжим исследование отображения
$$z_{n+1}=z_n^2+c.$$
В зелёном уровне мы построили общую картину множества Мандельброта. Теперь рассмотрим небольшой участок его границы и проверим, как результат зависит от разрешения сетки и числа итераций. Затем сравним способы окраски и посмотрим, как меняется множество Джулиа при пересечении этой границы.

Обозначим заполненное множество Джулиа через $K_c$, а его границу — через $J_c$. При построении Мандельброта меняется $c$ и берётся $z_0=0$; при построении $K_c$ параметр фиксирован, а по сетке меняется начальная точка $z_0$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm, Normalize
from matplotlib.patches import Rectangle, Patch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10
plt.rcParams["animation.embed_limit"] = 80


cmap = plt.get_cmap("turbo").copy()
cmap.set_bad("black")
no_escape = Patch(facecolor="black", label="Нет выхода за Nmax шагов")


## 1. Расчёт орбит

Используем алгоритм из зелёного уровня: на каждом шаге обновляем только те точки, которые ещё не вышли за радиус $R$. Для Мандельброта берём $R=2$, для Джулиа — $R=\max(2,|c|)$.

Для каждой точки сохраняем номер первого выхода $|z_n|>R$ и модуль $|z_n|$, который понадобится для плавной окраски. Если выхода за $N_{\max}$ шагов нет, записываем служебное значение $N_{\max}+1$ и показываем точку чёрным. Это ещё не означает, что её орбита останется ограниченной при дальнейшем расчёте.

Узлы сетки расположены в центрах пикселей. Для квадратного участка ширины $L$ и сетки $m\times m$ шаг по каждой оси равен $L/m$.


In [ ]:
def complex_grid(extent, size):
    xmin, xmax, ymin, ymax = extent
    x = xmin + (np.arange(size) + 0.5) * (xmax - xmin) / size
    y = ymin + (np.arange(size) + 0.5) * (ymax - ymin) / size
    return x[np.newaxis, :] + 1j * y[:, np.newaxis]


def escape_iterations(z0, c, Nmax, R=2):
    initial, parameters = np.broadcast_arrays(
        np.asarray(z0, dtype=complex), np.asarray(c, dtype=complex)
    )
    shape = initial.shape
    z = initial.reshape(-1).copy()
    c_values = parameters.reshape(-1)
    active = np.ones(z.size, dtype=bool)
    result = np.full(z.size, Nmax + 1, dtype=np.int32)
    exit_modulus = np.full(z.size, np.nan)

    initial_modulus = np.abs(z)
    escaped = initial_modulus > R
    result[escaped] = 0
    exit_modulus[escaped] = initial_modulus[escaped]
    active[escaped] = False

    for n in range(1, Nmax + 1):
        indices = np.flatnonzero(active)
        if indices.size == 0:
            break
        z[indices] = z[indices]**2 + c_values[indices]
        modulus = np.abs(z[indices])
        escaped_here = modulus > R
        exited = indices[escaped_here]
        result[exited] = n
        exit_modulus[exited] = modulus[escaped_here]
        active[exited] = False

    return result.reshape(shape), exit_modulus.reshape(shape)


def mandelbrot(extent, size, Nmax):
    return escape_iterations(0j, complex_grid(extent, size), Nmax, R=2)


def integer_colors(result, Nmax):
    return np.ma.masked_where(result > Nmax, result)


def draw_fractal(ax, values, extent, title, norm, plane="c"):
    image = ax.imshow(
        values, origin="lower", extent=extent, aspect="equal",
        cmap=cmap, norm=norm, interpolation="nearest"
    )
    ax.set_title(title)
    ax.set_xlabel(f"Re {plane}")
    ax.set_ylabel(f"Im {plane}")
    return image


## 2. Увеличение участка границы

Возьмём участок рядом с местом соединения основной кардиоиды и круглой компоненты периода 2:
$$-0.80\leq\operatorname{Re}c\leq-0.70,\qquad
0.05\leq\operatorname{Im}c\leq0.15.$$
На общей карте он отмечен белой рамкой. Ширина выбранного окна равна $0.1$, а исходного — $3$, поэтому увеличение составляет 30 раз. Для этого окна заново вычисляем орбиты на сетке $600\times600$.


In [ ]:
full_extent = [-2, 1, -1.5, 1.5]
zoom_extent = [-0.80, -0.70, 0.05, 0.15]
overview, overview_modulus = mandelbrot(full_extent, 600, 200)
zoom_preview, zoom_preview_modulus = mandelbrot(zoom_extent, 600, 200)

preview_norm = PowerNorm(gamma=0.4, vmin=0, vmax=200)
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
draw_fractal(ax[0], integer_colors(overview, 200), full_extent,
             "Общая карта: 600×600, Nmax=200", preview_norm)
image = draw_fractal(ax[1], integer_colors(zoom_preview, 200), zoom_extent,
                     "Новый расчёт выбранного окна: ×30", preview_norm)
ax[0].add_patch(Rectangle((zoom_extent[0], zoom_extent[2]), 0.1, 0.1,
                          fill=False, edgecolor="white", linewidth=1.5))
ax[0].annotate("Выбранное окно", (-0.75, 0.15), xytext=(-1.8, 0.8),
               color="white", arrowprops=dict(arrowstyle="->", color="white"))
bar = fig.colorbar(image, ax=ax, shrink=0.78, ticks=[0, 10, 25, 50, 100, 200])
bar.set_label("Первая итерация выхода n")
fig.legend(handles=[no_escape], loc="outside lower center")
plt.show()


## 3. Влияние сетки и числа итераций

Рассчитаем выбранный участок для сеток $600\times600$ и $1200\times1200$ при $N_{\max}=200$ и $800$. Сравнение строк покажет влияние разрешения, а сравнение столбцов — влияние числа итераций.

Для всех четырёх изображений оставим одну палитру и шкалу от 0 до 800. Используем `PowerNorm` с показателем $0.4$, чтобы лучше различать цвета при небольших номерах выхода. На цветовой шкале указаны сами номера итераций.


In [ ]:
sizes = [600, 1200]
limits = [200, 800]
results = {}

for size in sizes:
    for Nmax in limits:
        results[size, Nmax] = mandelbrot(zoom_extent, size, Nmax)

common_norm = PowerNorm(gamma=0.4, vmin=0, vmax=800)
fig, ax = plt.subplots(2, 2, figsize=(11, 10), layout="constrained")
fig.suptitle("Один участок границы: отдельное влияние сетки и Nmax")
for row, size in enumerate(sizes):
    for col, Nmax in enumerate(limits):
        result, _ = results[size, Nmax]
        image = draw_fractal(ax[row, col], integer_colors(result, Nmax), zoom_extent,
                             f"{size}×{size}, Nmax={Nmax}", common_norm)
bar = fig.colorbar(image, ax=ax, shrink=0.8,
                   ticks=[0, 10, 25, 50, 100, 200, 400, 800])
bar.set_label("Первая итерация выхода n; общая шкала")
fig.legend(handles=[no_escape], loc="outside lower center")
plt.show()


In [ ]:
print(" Сетка    Nmax    Шаг по осям     Без выхода, %    Выход на шагах 201–800, %")
for size in sizes:
    long_result = results[size, 800][0]
    delayed = np.count_nonzero((long_result > 200) & (long_result <= 800))
    for Nmax in limits:
        result = results[size, Nmax][0]
        pending = np.count_nonzero(result > Nmax)
        row = dict(size=size, Nmax=Nmax, step=0.1 / size,
                   unresolved=pending, unresolved_percent=100 * pending / result.size,
                   delayed_percent=100 * delayed / result.size)
        print(f"{size:5d}    {Nmax:4d}    {row['step']:.7f}       "
              f"{row['unresolved_percent']:8.4f}            {row['delayed_percent']:8.4f}")

    # На одной и той же сетке ранние выходы должны полностью совпасть.
    short_result = results[size, 200][0]
    early = short_result <= 200
    assert np.array_equal(short_result[early], long_result[early])
    assert np.count_nonzero(short_result > 200) - np.count_nonzero(long_result > 800) == delayed


for size in sizes:
    result = results[size, 800][0]
    delayed = np.count_nonzero((result > 200) & (result <= 800))
    print(f"На сетке {size}×{size} из чёрной области при Nmax=200 "
          f"ушли {delayed} точек ({100 * delayed / result.size:.4f}% всей сетки).")


При удвоении разрешения шаг сетки уменьшается вдвое. Это позволяет различать более мелкие детали границы, но каждая орбита по-прежнему прослеживается только до выбранного $N_{\max}$.

Увеличение числа итераций даёт другой эффект: часть чёрных точек успевает выйти за радиус и получает цвет. На сетке $600\times600$ доля точек без выхода уменьшилась с 74.2183% до 71.5200%, на сетке $1200\times1200$ — с 74.2171% до 71.5090%. В обоих случаях поздние выходы составили около 2.7% всех точек участка.

Таким образом, более густая сетка не заменяет более долгого расчёта орбит. В таблице мы сравниваем доли точек; напрямую сопоставлять пиксели двух разрешений нельзя, поскольку центры их ячеек не совпадают.


## 4. Плавная окраска

При окраске целым номером выхода $n$ соседние области с разными номерами образуют полосы. Чтобы сгладить переходы, учтём также модуль орбиты в момент выхода:
$$\nu=n-\frac{\ln\!\left(\ln|z_n|/\ln R\right)}{\ln2},\qquad |z_n|>R.$$

Эту формулу можно получить из приближения $|z_{n+1}|\approx|z_n|^2$, когда слагаемое $c$ уже мало по сравнению с $z_n^2$. Тогда $\ln|z|$ примерно удваивается за шаг. Продолжая эту зависимость на дробное число шагов назад до радиуса $R$, получаем оценку $\nu$. Чем сильнее точка превысила радиус при выходе, тем меньше эта оценка по сравнению с целым $n$.

Сравним обе окраски на уже рассчитанной сетке $1200\times1200$ при $N_{\max}=800$. Для двух панелей выберем одинаковую шкалу от 10 до 80, чтобы полосы были лучше видны. Значения за её пределами показаны крайними цветами.


In [ ]:
def smooth_colors(result, exit_modulus, Nmax, R=2):
    escaped = result <= Nmax
    values = np.zeros(result.shape, dtype=float)
    values[escaped] = result[escaped] - np.log2(
        np.log(exit_modulus[escaped]) / np.log(R)
    )
    return np.ma.array(values, mask=~escaped)


result, modulus = results[1200, 800]
smooth_result = smooth_colors(result, modulus, 800)
# Сужаем только цветовую шкалу, чтобы отчётливее различались полосы.
coloring_norm = Normalize(vmin=10, vmax=80, clip=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 5.2), layout="constrained")
fig.suptitle("Одна сетка 1200×1200, Nmax=800; одинаковая палитра и шкала")
draw_fractal(ax[0], integer_colors(result, 800), zoom_extent,
             "Обычная окраска: целое n", coloring_norm)
image = draw_fractal(ax[1], smooth_result, zoom_extent,
                     "Плавная окраска: дробное ν", coloring_norm)
bar = fig.colorbar(image, ax=ax, shrink=0.8,
                   ticks=[10, 20, 30, 40, 60, 80], extend="both")
bar.set_label("Число итераций: n или ν")
fig.legend(handles=[no_escape], loc="outside lower center")
plt.show()

assert np.array_equal(np.ma.getmaskarray(smooth_result), result > 800)
assert np.isfinite(smooth_result.compressed()).all()


При плавной окраске цвет меняется и внутри полосы с одним номером выхода, поэтому переходы выглядят мягче. Чёрная область остаётся прежней: обе картинки построены по одним и тем же орбитам, меняется только способ назначения цвета.


## 5. Переход через границу Мандельброта

Для анимации выберем отрезок вещественной оси:
$$c(t)=0.05+0.40t,\qquad 0\leq t\leq1.$$
Здесь $t$ задаёт положение на пути. При $t=1/2$ параметр равен $1/4$ — это точка границы основной кардиоиды.

Проверим, что путь действительно пересекает границу. При $0\leq c\leq1/4$ существует неподвижная точка
$$\alpha=\frac{1-\sqrt{1-4c}}2.$$
Орбита из нуля монотонно растёт и остаётся в отрезке $[0,\alpha]$. При $c>1/4$, напротив,
$$x_{n+1}-x_n=(x_n-1/2)^2+c-1/4>0.$$
Такая орбита не может быть ограниченной: иначе её предел был бы вещественной неподвижной точкой, а при $c>1/4$ их нет.

Возьмём 81 кадр, чтобы в середину последовательности попало $c=1/4$. Для проверки отдельно рассчитаем критическую орбиту $z_0=0$ до первого выхода или до 5000 шагов. В таблице и на графиках покажем несколько значений до, на и после границы.


In [ ]:
frame_t = np.linspace(0, 1, 81)
c_path = (0.05 + 0.40 * frame_t).astype(complex)
c_path[40] = 0.25 + 0j
critical_Nmax = 5000


def critical_orbit(c, Nmax=5000):
    R = max(2, abs(c))
    orbit = [0j]
    z = 0j
    for n in range(1, Nmax + 1):
        z = z**2 + c
        orbit.append(z)
        if abs(z) > R:
            return np.asarray(orbit), n
    return np.asarray(orbit), None


critical_results = [critical_orbit(c, critical_Nmax) for c in c_path]
selected_frames = [0, 20, 40, 41, 50, 80]
print("Кадр     c       Первый выход     max |z_n|       Последнее Re z_n")
for frame in selected_frames:
    c = c_path[frame]
    orbit, first_exit = critical_results[frame]
    status = str(first_exit) if first_exit is not None else f"нет за {critical_Nmax}"
    print(f"{frame:3d}    {c.real:.3f}    {status:>12s}       "
          f"{np.max(np.abs(orbit)):.6f}          {orbit[-1].real:.6f}")


fig, ax = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for frame in selected_frames:
    orbit, first_exit = critical_results[frame]
    label = f"кадр {frame}, c={c_path[frame].real:.3f}"
    indices = np.arange(min(len(orbit), 121))
    target = ax[0] if first_exit is None else ax[1]
    target.plot(indices, np.abs(orbit[:len(indices)]), label=label)
ax[0].set_title("Орбиты без выхода: первые 120 шагов")
ax[1].set_title("Орбиты с выходом: до первого |z| > 2")
ax[0].axhline(0.5, linestyle=":", color="gray", label="z = 1/2 при c = 1/4")
ax[1].axhline(2, linestyle="--", color="black", label="R = 2")
for a in ax:
    a.set_xlabel("Номер итерации n")
    a.set_ylabel("|z_n|, z₀ = 0")
    a.grid(True, alpha=0.3)
    a.legend(fontsize=8)
plt.show()


При $c=0.05$ и $0.15$ орбита приближается к неподвижной точке. На границе, при $c=0.25$, она стремится к $1/2$ заметно медленнее. Для $c=0.255$, $0.30$ и $0.45$ получен выход за радиус 2. Ближайшему к границе внешнему параметру $c=0.255$ требуется больше шагов до выхода.

Для первых трёх параметров ограниченность следует из рассуждения выше. Один лишь расчёт без выхода за 5000 шагов такого вывода не дал бы.


## 6. Анимация изменения множества Джулиа

Слева на карте Мандельброта движется точка $c$, справа показано соответствующее заполненное множество Джулиа. Под панелями указаны значение параметра и результат проверки его критической орбиты.

Для карты используем сетку $700\times700$, для кадров Джулиа — $480\times480$. Во всех кадрах сохраняем масштаб осей и цветовую шкалу, число итераций равно 400. Анимация встроена в ноутбук; её можно запустить кнопкой воспроизведения или просмотреть по кадрам с помощью ползунка.


In [ ]:
parameter_extent = [-0.6, 0.6, -0.6, 0.6]
julia_extent = [-1.5, 1.5, -1.5, 1.5]
animation_size = 480
animation_Nmax = 400
fps = 12
animation_norm = PowerNorm(gamma=0.4, vmin=0, vmax=animation_Nmax)

parameter_result, parameter_modulus = mandelbrot(parameter_extent, 700, animation_Nmax)
parameter_colors = smooth_colors(parameter_result, parameter_modulus, animation_Nmax)
z_grid = complex_grid(julia_extent, animation_size)
julia_frames = []
for frame, c in enumerate(c_path):
    R = max(2, abs(c))
    result, modulus = escape_iterations(z_grid, c, animation_Nmax, R)
    # float32 достаточно только для цветов; сами орбиты считаем в complex128.
    julia_frames.append(smooth_colors(result, modulus, animation_Nmax, R).astype(np.float32))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.4, 5.2))
fig.subplots_adjust(left=0.07, right=0.89, bottom=0.19, top=0.86, wspace=0.30)
fig.suptitle("Изменение множества Джулиа при пересечении границы Мандельброта", fontsize=12)
draw_fractal(ax[0], parameter_colors, parameter_extent,
             "Карта параметров: 700×700", animation_norm)
ax[0].plot(c_path.real, c_path.imag, color="white", linewidth=1.2)
ax[0].annotate("", xy=(0.45, 0.0), xytext=(0.34, 0.0),
               arrowprops=dict(arrowstyle="->", color="white", lw=1.5))
ax[0].plot([0.25], [0], "+", color="cyan", markersize=11, markeredgewidth=1.5)
ax[0].annotate("c = 1/4", (0.25, 0), xytext=(0.07, 0.14), color="white",
               arrowprops=dict(arrowstyle="-", color="white"), fontsize=9)
marker, = ax[0].plot([], [], "o", color="gold", markeredgecolor="black", markersize=7)
julia_image = draw_fractal(ax[1], julia_frames[0], julia_extent,
                           "Заполненное множество Джулиа: 480×480", animation_norm, plane="z₀")
cax = fig.add_axes([0.92, 0.24, 0.018, 0.53])
bar = fig.colorbar(julia_image, cax=cax, ticks=[0, 10, 25, 50, 100, 200, 400])
bar.set_label("Плавное число итераций ν", fontsize=9)
frame_label = fig.text(0.07, 0.10, "", fontsize=10)
critical_label = fig.text(0.07, 0.055, "", fontsize=9)
fig.text(0.07, 0.015, "Обе панели: Nmax=400, R=2. Чёрный: нет выхода за 400 шагов.", fontsize=9)


def update(frame):
    c = c_path[frame]
    marker.set_data([c.real], [c.imag])
    julia_image.set_data(julia_frames[frame])
    frame_label.set_text(f"Кадр {frame:02d}/80; t={frame_t[frame]:.4f}; c={c.real:.3f} + {c.imag:.3f}i")
    orbit, first_exit = critical_results[frame]
    if first_exit is None:
        status = f"нет выхода за {critical_Nmax} шагов; |z_last|={abs(orbit[-1]):.6f}"
    else:
        status = f"выход на шаге {first_exit}; |z_exit|={abs(orbit[-1]):.6f}"
    critical_label.set_text("Критическая орбита z₀=0: " + status)
    return marker, julia_image, frame_label, critical_label


animation = FuncAnimation(fig, update, frames=len(c_path), interval=1000 / fps, blit=False)


plt.close(fig)

# HTML содержит сами PNG-кадры в base64, а не ссылки на файлы.
animation_html = animation.to_jshtml(fps=fps, default_mode="once")

# Заменяем внешнюю библиотеку значков обычными символами:
# управление не требует загрузки CSS или шрифтов из интернета.
import re
animation_html = re.sub(r'<link\b[^>]*font-awesome[^>]*>', '', animation_html,
                        flags=re.DOTALL)
button_icons = {
    "fa fa-minus": "−", "fa fa-fast-backward": "⏮",
    "fa fa-step-backward": "⏪", "fa fa-play fa-flip-horizontal": "◀",
    "fa fa-pause": "⏸", "fa fa-play": "▶",
    "fa fa-step-forward": "⏩", "fa fa-fast-forward": "⏭",
    "fa fa-plus": "+",
}
for css_class, symbol in button_icons.items():
    animation_html = animation_html.replace(
        f'<i class="{css_class}"></i>', f'<span>{symbol}</span>'
    )
display(HTML(animation_html))


## 7. Связность множества Джулиа

**Как связаны ограниченность критической орбиты и связность множества Джулиа?**

Для отображения $f_c(z)=z^2+c$ критическая точка — $z=0$, поскольку $f'_c(z)=2z$. Если её орбита ограничена, то $K_c$ и его граница $J_c$ связны; такие параметры входят в множество Мандельброта. Если критическая орбита уходит на бесконечность, множество Джулиа вполне несвязно: каждая его связная компонента состоит из одной точки.

Поэтому на выбранном пути при $c\leq1/4$ множества связны, а после $c=1/4$ — вполне несвязны. Вблизи границы это изменение трудно различить на изображении.

**Что мешает определить связность по картинке?**

Тонкая перемычка может оказаться меньше шага сетки и исчезнуть на рисунке. Возможна и обратная ситуация: промежуток между отдельными частями множества слишком мал, и они выглядят соединёнными. Кроме того, при конечном $N_{\max}$ чёрными остаются и точки, которые выйдут позже, что мы увидели при сравнении 200 и 800 итераций.

Таким образом, по виду соседних пикселей нельзя установить связность. Для нашего пути вывод основан на поведении критической орбиты и его обосновании в разделе 5. Расчёты показывают, как переход выглядит при выбранном разрешении и числе итераций.
